# OOF reranker report

Upload or locate `oof_span_reranker_artifacts.zip`. This notebook performs no training or inference and does not require the dataset.

## Step 1 — Load and validate the artifact


In [1]:
import json,csv,io,zipfile,sys,collections
from pathlib import Path
from IPython.display import display,Markdown
NAME="oof_span_reranker_artifacts.zip"; paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_reranker_oof")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
 from google.colab import drive; drive.mount("/content/drive"); archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
 from google.colab import files; up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); assert zf.testzip() is None
summary=json.loads(zf.read("oof_reranker_results/oof_reranker_summary.json")); rows=list(csv.DictReader(io.StringIO(zf.read("oof_reranker_results/paired_deltas.csv").decode())))
print("Loaded",archive,"source",summary["source_run"],"— no training or inference")


Loaded oof_span_reranker_artifacts.zip source CurrentGated-s1-20261014-175436 — no training or inference


## Step 2 — Compare OOF reranker metrics


In [2]:
metrics=("bleu1","meteor","rougeL","em","f1"); lines=["| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |","|---|---:|---:|---:|---:|---:|"]
for name in ("BaseTop1","OOFLinear","OOFLambdaMART","Top10Oracle"):
 v=summary["scores"][name]; lines.append(f"| {name} | "+" | ".join(f"{v[m]:.2f}" for m in metrics)+" |")
display(Markdown("## Validation results\n\n"+"\n".join(lines)))


## Validation results

| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|
| BaseTop1 | 39.06 | 33.69 | 44.14 | 27.32 | 44.22 |
| OOFLinear | 39.81 | 32.35 | 44.78 | 29.83 | 45.08 |
| OOFLambdaMART | 40.60 | 34.26 | 45.56 | 28.90 | 45.73 |
| Top10Oracle | 62.02 | 51.33 | 67.16 | 49.16 | 67.15 |

## Step 3 — Run paired bootstrap analysis


In [3]:
import numpy as np
rng=np.random.default_rng(4012); lines=["| Model vs base | Metric | Mean Δ | 95% bootstrap CI | P(Δ≤0) | Win / tie / loss |","|---|---|---:|---:|---:|---:|"]
for prefix,label in (("linear","OOF Linear"),("tree","OOF LambdaMART")):
 for metric in ("rougeL","f1","em"):
  d=np.array([float(r[f"{prefix}_{metric}"]) for r in rows]); means=np.array([d[rng.integers(0,len(d),len(d))].mean() for _ in range(10000)])*100; lo,hi=np.quantile(means,[.025,.975])
  lines.append(f"| {label} | {metric} | {d.mean()*100:+.2f} | [{lo:+.2f}, {hi:+.2f}] | {(means<=0).mean():.3f} | {(d>1e-12).sum()} / {(abs(d)<=1e-12).sum()} / {(d< -1e-12).sum()} |")
display(Markdown("## Paired bootstrap\n\n"+"\n".join(lines)))


## Paired bootstrap

| Model vs base | Metric | Mean Δ | 95% bootstrap CI | P(Δ≤0) | Win / tie / loss |
|---|---|---:|---:|---:|---:|
| OOF Linear | rougeL | +0.64 | [-0.25, +1.55] | 0.085 | 59 / 970 / 47 |
| OOF Linear | f1 | +0.85 | [-0.04, +1.73] | 0.031 | 58 / 971 / 47 |
| OOF Linear | em | +2.51 | [+1.49, +3.53] | 0.000 | 29 / 1045 / 2 |
| OOF LambdaMART | rougeL | +1.42 | [+0.28, +2.55] | 0.008 | 94 / 916 / 66 |
| OOF LambdaMART | f1 | +1.50 | [+0.36, +2.69] | 0.004 | 95 / 916 / 65 |
| OOF LambdaMART | em | +1.58 | [+0.37, +2.79] | 0.004 | 30 / 1033 / 13 |

## Step 4 — Inspect validation slices


In [4]:
for field,title in (("question_type","Question type"),("silver_region","Silver-span region")):
 groups=collections.defaultdict(list)
 for r in rows: groups[r[field]].append(r)
 lines=[f"| {title} | N | OOF Linear ΔRL | OOF LambdaMART ΔRL |","|---|---:|---:|---:|"]
 for name,v in sorted(groups.items()): lines.append(f"| {name} | {len(v)} | {sum(float(x['linear_rougeL']) for x in v)*100/len(v):+.2f} | {sum(float(x['tree_rougeL']) for x in v)*100/len(v):+.2f} |")
 display(Markdown(f"## {title} effects\n\n"+"\n".join(lines)))


## Question type effects

| Question type | N | OOF Linear ΔRL | OOF LambdaMART ΔRL |
|---|---:|---:|---:|
| how | 85 | +0.82 | -1.31 |
| other | 95 | -0.09 | +1.09 |
| what | 462 | +1.07 | +2.23 |
| when | 24 | -1.83 | -2.99 |
| where | 75 | +0.06 | -0.81 |
| which | 22 | +1.32 | +0.00 |
| who | 290 | +0.77 | +2.40 |
| why | 23 | -3.59 | -2.35 |

## Silver-span region effects

| Silver-span region | N | OOF Linear ΔRL | OOF LambdaMART ΔRL |
|---|---:|---:|---:|
| body | 820 | +0.86 | +1.97 |
| byline | 76 | -0.44 | -2.56 |
| cross | 18 | +1.03 | +5.97 |
| no_span | 162 | +0.00 | +0.00 |

## Step 5 — Inspect ranks and feature importance


In [5]:
for model in ("OOFLinear","OOFLambdaMART"):
 counts={int(k):v for k,v in summary["selected_rank_counts"][model].items()}; total=sum(counts.values())
 display(Markdown(f"## {model} selected ranks\n\n"+"\n".join(["| Base rank | Count | Share |","|---:|---:|---:|"]+[f"| {k} | {counts.get(k,0)} | {100*counts.get(k,0)/total:.1f}% |" for k in range(1,11)])))
lines=["| Feature | OOF LambdaMART importance |","|---|---:|"]+[f"| {n} | {v} |" for n,v in summary["tree_importances"]]
display(Markdown("## Feature importance\n\n"+"\n".join(lines)))


## OOFLinear selected ranks

| Base rank | Count | Share |
|---:|---:|---:|
| 1 | 925 | 86.0% |
| 2 | 101 | 9.4% |
| 3 | 27 | 2.5% |
| 4 | 9 | 0.8% |
| 5 | 5 | 0.5% |
| 6 | 4 | 0.4% |
| 7 | 1 | 0.1% |
| 8 | 1 | 0.1% |
| 9 | 2 | 0.2% |
| 10 | 1 | 0.1% |

## OOFLambdaMART selected ranks

| Base rank | Count | Share |
|---:|---:|---:|
| 1 | 821 | 76.3% |
| 2 | 92 | 8.6% |
| 3 | 39 | 3.6% |
| 4 | 29 | 2.7% |
| 5 | 20 | 1.9% |
| 6 | 20 | 1.9% |
| 7 | 12 | 1.1% |
| 8 | 15 | 1.4% |
| 9 | 20 | 1.9% |
| 10 | 8 | 0.7% |

## Feature importance

| Feature | OOF LambdaMART importance |
|---|---:|
| context_jaccard | 228 |
| base_score | 192 |
| start_fraction | 182 |
| length_fraction | 160 |
| end_logp | 158 |
| length | 151 |
| start_logp | 141 |
| question_jaccard | 126 |
| score_margin | 102 |
| distance_from_end | 93 |
| end_fraction | 79 |
| context_overlap | 75 |
| qtype_what | 57 |
| qtype_who | 53 |
| question_overlap | 48 |
| qtype_why | 46 |
| ends_word | 30 |
| reciprocal_rank | 29 |
| starts_word | 24 |
| qtype_other | 18 |
| region_byline | 16 |
| contains_mention | 11 |
| contains_number | 10 |
| qtype_where | 9 |
| qtype_how | 8 |
| qtype_when | 6 |
| region_body | 3 |
| qtype_which | 2 |
| region_cross | 1 |
| contains_url | 0 |

## Step 6 — Write the conclusion


In [6]:
best=max(("OOFLinear","OOFLambdaMART"),key=lambda n:summary["scores"][n]["rougeL"]); d=summary["delta_vs_base"][best]
text=f"""# Three-fold OOF span reranker

The stronger OOF reranker was **{best}**, changing validation ROUGE-L by **{d['rougeL']:+.2f}**,
F1 by **{d['f1']:+.2f}**, and EM by **{d['em']:+.2f}** versus unchanged Top-1 decoding.
Every reranker-training candidate was generated by a base model that had not seen that question or tweet.
Validation labels were used only for evaluation, and the locked test set was not accessed.
"""
Path("oof_span_reranker_report.md").write_text(text,encoding="utf-8"); display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Three-fold OOF span reranker

The stronger OOF reranker was **OOFLambdaMART**, changing validation ROUGE-L by **+1.42**,
F1 by **+1.50**, and EM by **+1.58** versus unchanged Top-1 decoding.
Every reranker-training candidate was generated by a base model that had not seen that question or tweet.
Validation labels were used only for evaluation, and the locked test set was not accessed.
